# Group Project Part 2: Discovering Patterns in SLCM Community Needs
**CS430: Starter notebook**

Team members: Aidan Cissell and Jayden West

This notebook only loads the prepared files and shows where they came from. Every analysis decision is your team's. Copy each task from the handout into a Markdown cell and answer below it, as in Assignments 1–3.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram

## 2. Load the prepared files
`zip_code` must be read as text, otherwise leading digits and ZIP meaning are lost.

In [ ]:
profiles = pd.read_csv("SLCM_part2_intake_profiles_clean.csv", dtype={"zip_code": str})
members  = pd.read_csv("SLCM_part2_household_members_clean.csv")
subs     = pd.read_csv("SLCM_part2_care_path_subtickets.csv")
dictionary = pd.read_csv("SLCM_part2_data_dictionary.csv")
cleaning_log = pd.read_csv("SLCM_part2_cleaning_log.csv")

print("Profiles:", profiles.shape, "| Members:", members.shape, "| Sub-tickets:", subs.shape)
print("Profiles + sub-tickets =", len(profiles) + len(subs), "(original masked file had 639 rows)")

## 3. Read the data dictionary and cleaning log first
The `role` column tells you which columns are candidate need features, which are context for profiling, which are data-quality flags, and which must not be used as clustering inputs.

In [ ]:
dictionary["role"].value_counts()

In [ ]:
pd.set_option("display.max_colwidth", 120)
dictionary[["column", "role", "description", "pct_missing"]]

In [ ]:
cleaning_log[["step", "field", "rule", "rows_affected"]]

## Extra setup

In [ ]:
# extra imports and settings used below
import os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

seed = 42
os.makedirs("figures", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# role of each column from the data dictionary
roles = dict(zip(dictionary["column"].astype(str), dictionary["role"].astype(str).str.lower()))

In [ ]:
# full list of columns with their dictionary role (no row values)
print(dictionary[["column", "role", "pct_missing"]].to_string())

In [ ]:
# type, missingness and number of distinct values for every profile column
structure = pd.DataFrame({
    "dtype": profiles.dtypes.astype(str),
    "missing_pct": (profiles.isna().mean() * 100).round(1),
    "n_unique": profiles.nunique()
})
print(structure.to_string())

## Task 0: Revisit Part 1 and define the analysis unit

**1.** Briefly recap two important Part 1 findings and one data-quality limitation. Explain how they influence your Part 2 approach.

**Answer:** _Two things we found in Part 1 are that employed families still rely on SNAP benefits even though they are receiving income from work, and that only 27 out of 150 repeat people had housing recorded more than once. Because a job alone did not mean a family was stable, employment status and income are both clustering inputs in Part 2. Because housing was rarely recorded more than once, there is one row per person instead of one row per ticket, so repeat visitors aren’t counted twice. One limitation was extreme data, like a $125,100 rent entry. In Part 2, income goes on a log scale and $0 gets its own flag, so a few huge values or zero values don’t control the clusters._

**2.** Define your observational unit (one ticket, one person, or another defensible unit). Describe how you treat Issue Type, tickets flagged is_duplicate_ticket, repeat PID values (see pid_ticket_seq and pid_is_latest_profile) and records missing core fields. Start from 426 profile tickets and 395 PIDs, report the number of observations that remain, and do not silently drop records.

In [ ]:
# starting counts: profile tickets and distinct people
print("profile tickets:", len(profiles))
print("distinct issue keys:", profiles["issue_key"].nunique())
print("distinct pids:", profiles["pid"].nunique())
print("missing pid:", profiles["pid"].isna().sum())

In [ ]:
# helper that reads true/false flags stored as text, numbers or booleans
def to_bool(series):
    return series.astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes", "y"])


# ticket types and duplicate flags
print(profiles["issue_type"].value_counts(dropna=False))
print()
print(profiles["is_duplicate_ticket"].value_counts(dropna=False))
print()
print(pd.crosstab(profiles["issue_type"], to_bool(profiles["is_duplicate_ticket"]), margins=True))

In [ ]:
# repeat contact: how many tickets each person has, and the latest-profile flag
tickets_per_pid = profiles.groupby("pid")["issue_key"].count()
print("tickets per pid:")
print(tickets_per_pid.value_counts().sort_index())
print()
print("pid_ticket_seq:")
print(profiles["pid_ticket_seq"].value_counts(dropna=False).sort_index())
print()
print("pid_is_latest_profile:")
print(profiles["pid_is_latest_profile"].value_counts(dropna=False))

if "pid_n_profile_tickets" in profiles.columns:
    print()
    print("pid_n_profile_tickets:")
    print(profiles["pid_n_profile_tickets"].value_counts(dropna=False).sort_index())

**Unit decision.** Set `unit` below. `"person"` keeps one row per PID (that person's latest non-duplicate profile),
so people with repeat contacts are not counted more than once. `"ticket"` keeps every non-duplicate profile.
Tickets with no PID are kept as their own unit either way. The number of profiles per person is kept as
`n_profiles` to describe clusters (Question 11), never as a clustering input.

In [ ]:
# build the analysis table and record the row count after each step
unit = "person"

flow = []
flow.append(("profile tickets loaded", len(profiles)))

work = profiles.copy()
work["dup_flag"] = to_bool(work["is_duplicate_ticket"])
work = work[~work["dup_flag"]].copy()
flow.append(("removed tickets flagged is_duplicate_ticket", len(work)))

# people with no pid keep their own ticket as the unit
work["unit_id"] = work["pid"].astype(str)
no_pid = work["pid"].isna()
work.loc[no_pid, "unit_id"] = "ticket_" + work.loc[no_pid, "issue_key"].astype(str)

# number of profiles each person has (describe only)
contacts = work.groupby("unit_id")["issue_key"].count().rename("n_profiles")
work = work.join(contacts, on="unit_id")

if unit == "person":
    work["seq_num"] = pd.to_numeric(work["pid_ticket_seq"], errors="coerce").fillna(1)
    work = work.sort_values(["unit_id", "seq_num"])
    analysis = work.groupby("unit_id").tail(1).copy()
    flow.append(("kept latest profile per person", len(analysis)))
else:
    analysis = work.copy()

analysis = analysis.reset_index(drop=True)
print(pd.DataFrame(flow, columns=["step", "rows"]).to_string(index=False))
print()
print("units with more than one profile:", (analysis["n_profiles"] > 1).sum())

**Answer:** _We used one row per person because repeat contacts would otherwise be counted more than once and pull the clusters toward those people. We started with 426 profile tickets and 395 PIDs. 12 tickets flagged as duplicates were removed, leaving 414, and then we kept each person’s latest profile, leaving 389. 21 people had more than one profile, and that count is only used to describe the clusters in Question 3, not as a clustering input. Issue Type wasn’t used as an input because it reflects where the person is in SLCM’s process, not their needs. There were 0 people missing all core fields, so nothing was dropped at that step. The final count was 389 people._

## Task 1: Prepare features for unsupervised learning

**3.** Choose a small, justified set of features describing needs. Possible candidates include housing status, eviction status, employment status, income amount, number of children/household members, past-due rent, and ZIP code. Do not use Issue key, PID or other identifiers as clustering inputs. Use sensitive fields only to describe clusters, never to define them.

The code below searches the column names for each candidate feature and prints what it picked.
**Check the printout.** If a pick is wrong, put the correct column name in `manual` and re-run from here.
Past-due rent is only recorded for about 38% of profiles because it comes from a separate rent form, so by default
it is used to *describe* clusters, not to build them (`use_past_due = False`). Otherwise the clusters would
mostly split on "filled out the rent form or not".

In [ ]:
# candidate need features: search keys in priority order, and words that rule a column out
spec = {
    "housing_status": {"kind": "cat", "keys": ["housing_status", "housing status", "housing"], "exclude": ["proxy", "urgent", "section", "voucher", "flag"]},
    "eviction_status": {"kind": "cat", "keys": ["eviction_status", "eviction status", "evict"], "exclude": ["proxy", "urgent", "flag", "date"]},
    "employment_status": {"kind": "cat", "keys": ["employment_status", "employment status", "employment", "employ"], "exclude": ["employer", "flag"]},
    "income_amount": {"kind": "num", "keys": ["income_amount", "monthly_income", "income amount", "income"], "exclude": ["source", "src", "causation", "loss", "flag", "annual", "zero", "missing", "is_"]},
    "household_size": {"kind": "num", "keys": ["household_size", "household size", "additional_household", "additional household", "household_members", "household"], "exclude": ["flag", "more_than", "age"]},
    "num_children": {"kind": "num", "keys": ["num_children", "number_of_children", "children", "child"], "exclude": ["flag", "age", "more_than"]},
    "past_due_rent": {"kind": "num", "keys": ["past_due_rent", "past_due", "past due", "arrears"], "exclude": ["high_arrears", "3000", "flag", "missing", "recorded", "utility"]}
}

# put exact column names here to override a wrong automatic pick, e.g. {"income_amount": "monthly_income_clean"}
manual = {}

use_past_due = False

blocked_roles = ["ident", "sensitive", "quality", "flag"]


def role_blocked(col):
    role = roles.get(col, "")
    if len(roles) > 0 and "need" not in role:
        return True
    return any(b in role for b in blocked_roles)


def mostly_numeric(col):
    present = profiles[col].notna().sum()
    if present == 0:
        return False
    raw = profiles[col].replace({"More than 5": 6, "more than 5": 6})
    vals = pd.to_numeric(raw, errors="coerce")
    return vals.notna().sum() / present >= 0.8


def candidates(name):
    found = []
    for key in spec[name]["keys"]:
        for col in profiles.columns:
            low = col.lower()
            if key in low and col not in found:
                if not any(e in low for e in spec[name]["exclude"]) and not role_blocked(col):
                    found.append(col)
    return found


chosen = {}
rows = []
for name in spec:
    found = candidates(name)
    pick = None
    if name in manual:
        pick = manual[name]
    else:
        for col in found:
            numeric = mostly_numeric(col)
            if spec[name]["kind"] == "num" and numeric:
                pick = col
                break
            elif spec[name]["kind"] == "cat" and not numeric and profiles[col].nunique() <= 30:
                pick = col
                break
    chosen[name] = pick
    rows.append({"feature": name, "kind": spec[name]["kind"], "picked column": pick, "other candidates": [c for c in found if c != pick][:6]})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# copy the picked columns into clean feature columns; unknown categories stay "Unknown", not stable or at risk
num_features = []
cat_features = []
for name, col in chosen.items():
    if col is None:
        continue
    if spec[name]["kind"] == "num":
        raw = analysis[col].replace({"More than 5": 6, "more than 5": 6})
        analysis[name] = pd.to_numeric(raw, errors="coerce")
        num_features.append(name)
    else:
        vals = analysis[col].where(analysis[col].notna(), "Unknown").astype(str).str.strip()
        vals = vals.replace({"": "Unknown", "nan": "Unknown", "None": "Unknown"})
        analysis[name] = vals
        cat_features.append(name)

print("numeric:", num_features)
print("categorical:", cat_features)

In [ ]:
# do not silently drop: report units missing every core field, then remove them
core = [f for f in ["housing_status", "employment_status", "income_amount"] if f in num_features + cat_features]
missing_core = pd.Series(True, index=analysis.index)
for f in core:
    if f in num_features:
        missing_core = missing_core & analysis[f].isna()
    else:
        missing_core = missing_core & (analysis[f] == "Unknown")

print("core fields:", core)
print("units missing all core fields:", missing_core.sum())
analysis = analysis[~missing_core].reset_index(drop=True)
flow.append(("removed units missing all core need fields", len(analysis)))
print(pd.DataFrame(flow, columns=["step", "rows"]).to_string(index=False))

**Answer:** _Which features you chose and why; why ZIP, identifiers, sensitive fields, urgent_housing_proxy and high_arrears_3000 are not inputs._

**4.** Create a concise feature table: chosen field, what it represents, its missingness, preparation/encoding choice, and reason for inclusion. Investigate the effect of missing values, zeros and extreme values. Explain any encoding, scaling or transformation you use.

In [ ]:
# numeric features: missingness, zeros, spread and skew
num_rows = []
for f in num_features:
    s = analysis[f]
    num_rows.append({
        "feature": f,
        "source column": chosen[f],
        "missing_pct": round(s.isna().mean() * 100, 1),
        "zero_pct": round((s == 0).mean() * 100, 1),
        "min": s.min(),
        "median": s.median(),
        "mean": round(s.mean(), 1),
        "p99": s.quantile(0.99),
        "max": s.max(),
        "skew": round(s.skew(), 2)
    })
print(pd.DataFrame(num_rows).to_string(index=False))

In [ ]:
# categorical features: missingness and category counts
for f in cat_features:
    print(f, "(source:", chosen[f] + ")")
    print(analysis[f].value_counts())
    print()

In [ ]:
# distribution of income and past-due rent before and after a log transform
money_cols = [f for f in ["income_amount", "past_due_rent"] if f in num_features]
if len(money_cols) > 0:
    fig, axes = plt.subplots(len(money_cols), 2, figsize=(11, 3.5 * len(money_cols)), squeeze=False)
    for i, f in enumerate(money_cols):
        s = analysis[f].dropna()
        axes[i][0].hist(s, bins=30, color="#4C72B0")
        axes[i][0].set_title(f + " (raw)")
        axes[i][1].hist(np.log1p(s.clip(lower=0)), bins=30, color="#55A868")
        axes[i][1].set_title(f + " (log1p)")
    plt.tight_layout()
    plt.savefig("figures/fig_money_distributions.png", dpi=150)
    plt.show()

In [ ]:
# inputs: income on a log scale plus a zero flag, counts scaled, categories one-hot encoded
inputs_money = [f for f in ["income_amount"] if f in num_features]
inputs_count = [f for f in num_features if f not in ["income_amount", "past_due_rent"]]
inputs_flag = []
inputs_cat = list(cat_features)

if "income_amount" in num_features:
    analysis["income_is_zero"] = (analysis["income_amount"] == 0).astype(float)
    inputs_flag.append("income_is_zero")
    if analysis["income_amount"].isna().any():
        analysis["income_missing"] = analysis["income_amount"].isna().astype(float)
        inputs_flag.append("income_missing")

if use_past_due and "past_due_rent" in num_features:
    inputs_money.append("past_due_rent")
    analysis["past_due_missing"] = analysis["past_due_rent"].isna().astype(float)
    inputs_flag.append("past_due_missing")


def log1p_clip(x):
    return np.log1p(np.clip(x, 0, None))


# missing flags are kept as plain 0/1 columns so they do not outweigh the real need fields
money_pipe = Pipeline([
    ("log", FunctionTransformer(log1p_clip, feature_names_out="one-to-one")),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

count_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

try:
    onehot = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore")

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("onehot", onehot)
])

parts = []
if len(inputs_money) > 0:
    parts.append(("money", money_pipe, inputs_money))
if len(inputs_count) > 0:
    parts.append(("count", count_pipe, inputs_count))
if len(inputs_flag) > 0:
    parts.append(("flag", "passthrough", inputs_flag))
if len(inputs_cat) > 0:
    parts.append(("cat", cat_pipe, inputs_cat))

prep = ColumnTransformer(parts, sparse_threshold=0, remainder="drop")

# no target and no train/test split, so the preprocessor is fit on the whole analysis table
x = prep.fit_transform(analysis[inputs_money + inputs_count + inputs_flag + inputs_cat])
try:
    x_names = list(prep.get_feature_names_out())
except Exception:
    x_names = ["f" + str(i) for i in range(x.shape[1])]

print("clustering inputs:", inputs_money + inputs_count + inputs_flag + inputs_cat)
print("matrix shape:", x.shape)
print(x_names)

In [ ]:
# feature table for the documentation; fill in the reason column yourselves
encoding = {}
for f in inputs_money:
    encoding[f] = "log1p, median impute, standard scale (missing flag kept as 0/1)"
for f in inputs_count:
    encoding[f] = "'More than 5' = 6, median impute, standard scale"
for f in inputs_flag:
    encoding[f] = "0/1 flag, not scaled"
for f in inputs_cat:
    encoding[f] = "missing -> 'Unknown', one-hot (categories under 5 grouped)"
if "past_due_rent" in num_features and not use_past_due:
    encoding["past_due_rent"] = "not a clustering input; used to describe clusters"

feature_table = []
for f in list(encoding.keys()):
    src = chosen.get(f, "derived from income")
    miss = analysis[f].isna().mean() * 100 if f in num_features else (analysis[f] == "Unknown").mean() * 100 if f in cat_features else 0
    feature_table.append({"feature": f, "source column": src, "missing_or_unknown_pct": round(miss, 1), "preparation": encoding[f], "reason": ""})

feature_table = pd.DataFrame(feature_table)
feature_table.to_csv("outputs/feature_table.csv", index=False)
print(feature_table.to_string(index=False))

**Answer:** _What the income and past-due distributions show (zeros, skew, extremes); why log scale, the $0 flag, and keeping Unknown as its own category. Fill the reason column in the feature table._

**5.** Keep an original-to-cleaned row count and extend the provided cleaning log with your team's own decisions. Avoid presenting missing or "unknown" status as though it is confirmed housing stability or confirmed risk.

In [ ]:
# row-count flow from the original profile file to the analysis table
print(pd.DataFrame(flow, columns=["step", "rows"]).to_string(index=False))

In [ ]:
# extend the provided cleaning log with the team's decisions (same columns)
try:
    next_step = int(pd.to_numeric(cleaning_log["step"], errors="coerce").max()) + 1
except Exception:
    next_step = len(cleaning_log) + 1

team_rows = [
    {"field": "is_duplicate_ticket", "rule": "team: removed profile tickets flagged as duplicates", "rows_affected": flow[0][1] - flow[1][1]},
    {"field": "PID / pid_ticket_seq", "rule": "team: one row per person, kept latest non-duplicate profile; tickets without PID kept as their own unit", "rows_affected": flow[1][1] - flow[2][1]},
    {"field": ", ".join(core), "rule": "team: removed units missing all core need fields", "rows_affected": flow[2][1] - flow[3][1]},
    {"field": ", ".join(cat_features), "rule": "team: missing category kept as 'Unknown' (not treated as stable or at risk)", "rows_affected": int(sum((analysis[f] == "Unknown").sum() for f in cat_features))},
    {"field": "income_amount", "rule": "team: log1p transform; $0 kept as 0 with income_is_zero flag; missing median-imputed with income_missing flag", "rows_affected": int(analysis["income_amount"].isna().sum() + (analysis["income_amount"] == 0).sum()) if "income_amount" in num_features else 0},
    {"field": "past_due_rent", "rule": "team: not a clustering input (recorded only when rent form filled); used to describe clusters" if not use_past_due else "team: log1p transform with missing flag", "rows_affected": int(analysis["past_due_rent"].isna().sum()) if "past_due_rent" in num_features else 0},
    {"field": "identifiers, sensitive fields, urgent_housing_proxy, high_arrears_3000", "rule": "team: excluded from clustering inputs; used only to describe clusters", "rows_affected": 0}
]
for i, row in enumerate(team_rows):
    row["step"] = next_step + i

full_log = pd.concat([cleaning_log, pd.DataFrame(team_rows)], ignore_index=True)
full_log.to_csv("outputs/cleaning_log_extended.csv", index=False)
full_log[["step", "field", "rule", "rows_affected"]].tail(len(team_rows))

**Answer:** _One or two sentences pointing to the row-count flow and the extended log._

## Task 2: K-Means, hierarchical clustering, and cluster profiles

**6.** Apply K-Means clustering. Try a small range of reasonable cluster counts (for example k = 2–6), use an elbow plot and report silhouette scores. Select a cluster count and justify your choice with both evidence and interpretability.

In [ ]:
# elbow and silhouette for k = 2 to 8
k_results = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=20, random_state=seed)
    lab = km.fit_predict(x)
    k_results.append({
        "k": k,
        "inertia": round(km.inertia_, 1),
        "silhouette": round(silhouette_score(x, lab), 3),
        "smallest_cluster": int(np.bincount(lab).min()),
        "largest_cluster": int(np.bincount(lab).max())
    })
k_results = pd.DataFrame(k_results)
print(k_results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(k_results["k"], k_results["inertia"], marker="o")
axes[0].set_title("Elbow plot (K-Means)")
axes[0].set_xlabel("k")
axes[0].set_ylabel("inertia")
axes[1].plot(k_results["k"], k_results["silhouette"], marker="o", color="#C44E52")
axes[1].set_title("Silhouette score (K-Means)")
axes[1].set_xlabel("k")
axes[1].set_ylabel("silhouette")
plt.tight_layout()
plt.savefig("figures/fig_kmeans_elbow_silhouette.png", dpi=150)
plt.show()

**Choose k.** By default the code picks the best silhouette among k = 3–6 where no cluster is under 5% of units.
After looking at the elbow plot and the profiles, you can override it by setting `chosen_k` to a number.

In [ ]:
# pick k (override with a number if the plots and profiles support a different choice)
chosen_k = None

if chosen_k is None:
    ok = k_results[(k_results["k"].between(3, 6)) & (k_results["smallest_cluster"] >= 0.05 * len(analysis))]
    if len(ok) == 0:
        ok = k_results[k_results["k"].between(3, 6)]
    chosen_k = int(ok.sort_values("silhouette", ascending=False).iloc[0]["k"])
print("chosen k:", chosen_k)

kmeans = KMeans(n_clusters=chosen_k, n_init=50, random_state=seed)
km_labels = kmeans.fit_predict(x)
print("cluster sizes:", np.bincount(km_labels))
print("silhouette:", round(silhouette_score(x, km_labels), 3))

In [ ]:
# stability: refit with 10 different seeds and compare to the final labels
aris = []
for s in range(10):
    lab = KMeans(n_clusters=chosen_k, n_init=10, random_state=s).fit_predict(x)
    aris.append(adjusted_rand_score(km_labels, lab))
print("adjusted rand index across seeds: mean", round(np.mean(aris), 3), "min", round(np.min(aris), 3))

In [ ]:
# which inputs drive each cluster: k-means centers on the scaled inputs
centers = pd.DataFrame(kmeans.cluster_centers_, columns=x_names)
centers.index = ["Cluster " + str(i) for i in range(chosen_k)]

fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(x_names)), 0.6 * chosen_k + 6))
im = ax.imshow(centers.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2)
ax.set_xticks(range(len(x_names)))
ax.set_xticklabels([n.split("__", 1)[-1] for n in x_names], rotation=75, ha="right", fontsize=8)
ax.set_yticks(range(chosen_k))
ax.set_yticklabels(centers.index)
ax.set_title("K-Means cluster centers (scaled inputs; one-hot columns show share in cluster)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("figures/fig_kmeans_centers.png", dpi=150)
plt.show()
print(centers.round(2).T.to_string())

**Answer:** _Which k you chose, citing the elbow and silhouette, and why it is interpretable._

**7.** Apply hierarchical/agglomerative clustering to the same prepared data. Include a dendrogram for a readable sample or another appropriate visualization. Compare whether the broad patterns are similar to K-Means; explain differences rather than expecting identical labels.

In [ ]:
# ward linkage on the same prepared matrix; dendrogram of all units (top merges) with the cut for chosen_k
z = linkage(x, method="ward")
cut = (z[-chosen_k, 2] + z[-chosen_k + 1, 2]) / 2

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(z, truncate_mode="lastp", p=30, ax=ax, no_labels=False, leaf_font_size=8)
ax.axhline(cut, color="red", linestyle="--", label="cut for k = " + str(chosen_k))
ax.set_title("Ward dendrogram (all units, last 30 merges)")
ax.set_ylabel("merge distance")
ax.legend()
plt.tight_layout()
plt.savefig("figures/fig_dendrogram_full.png", dpi=150)
plt.show()

In [ ]:
# readable dendrogram on a random sample of 60 units
rng = np.random.default_rng(seed)
sample_idx = rng.choice(len(x), size=min(60, len(x)), replace=False)
z_sample = linkage(x[sample_idx], method="ward")

fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(z_sample, ax=ax, no_labels=True)
ax.set_title("Ward dendrogram (random sample of " + str(len(sample_idx)) + " units)")
ax.set_ylabel("merge distance")
plt.tight_layout()
plt.savefig("figures/fig_dendrogram_sample.png", dpi=150)
plt.show()

In [ ]:
# agglomerative clustering at the same k, relabeled to line up with the k-means clusters
hc_raw = AgglomerativeClustering(n_clusters=chosen_k, linkage="ward").fit_predict(x)

overlap = pd.crosstab(hc_raw, km_labels).reindex(index=range(chosen_k), columns=range(chosen_k), fill_value=0)
row_ind, col_ind = linear_sum_assignment(-overlap.values)
relabel = dict(zip(row_ind, col_ind))
hc_labels = np.array([relabel[v] for v in hc_raw])

print("hierarchical cluster sizes:", np.bincount(hc_labels, minlength=chosen_k))
print("hierarchical silhouette:", round(silhouette_score(x, hc_labels), 3))
print("k-means silhouette:", round(silhouette_score(x, km_labels), 3))
print("adjusted rand index (k-means vs hierarchical):", round(adjusted_rand_score(km_labels, hc_labels), 3))
print("units with the same cluster in both methods:", round((km_labels == hc_labels).mean() * 100, 1), "%")
print()
print(pd.crosstab(pd.Series(km_labels, name="k-means"), pd.Series(hc_labels, name="hierarchical"), margins=True))

In [ ]:
# silhouette of hierarchical clustering across k for comparison with k-means
hc_rows = []
for k in range(2, 9):
    lab = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(x)
    hc_rows.append({"k": k, "hierarchical_silhouette": round(silhouette_score(x, lab), 3)})
print(k_results[["k", "silhouette"]].merge(pd.DataFrame(hc_rows), on="k").rename(columns={"silhouette": "kmeans_silhouette"}).to_string(index=False))

In [ ]:
# 2-D PCA view of both clusterings (for visualization only)
pca = PCA(n_components=2, random_state=seed)
pts = pca.fit_transform(x)
print("variance explained by 2 components:", round(pca.explained_variance_ratio_.sum() * 100, 1), "%")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, lab, title in [(axes[0], km_labels, "K-Means"), (axes[1], hc_labels, "Hierarchical (Ward)")]:
    for c in range(chosen_k):
        mask = lab == c
        ax.scatter(pts[mask, 0], pts[mask, 1], s=14, alpha=0.7, label="Cluster " + str(c))
    ax.set_title(title + ", k = " + str(chosen_k))
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
axes[0].legend()
plt.tight_layout()
plt.savefig("figures/fig_pca_clusters.png", dpi=150)
plt.show()

**Answer:** _How similar the two methods are (adjusted Rand index, % same cluster, crosstab) and where they differ._

**8.** Profile each final cluster using understandable tables and plots: cluster size, important numerical summaries, and relevant category proportions. Give clusters neutral descriptive names based only on measured attributes, not stigmatizing labels.

Give each cluster a neutral name based only on measured attributes (e.g. "employed, $0 reported income, renting").
Fill in `cluster_names` after reading the profiles and re-run the cells below.

In [ ]:
# neutral cluster names (fill these in after reading the profiles)
cluster_names = {}

analysis["cluster"] = km_labels
analysis["cluster_hc"] = hc_labels
analysis["cluster_name"] = [cluster_names.get(c, "Cluster " + str(c)) for c in km_labels]

sizes = analysis["cluster_name"].value_counts().sort_index()
size_table = pd.DataFrame({"n": sizes, "pct": (sizes / len(analysis) * 100).round(1)})
print(size_table.to_string())

In [ ]:
# numeric summaries by cluster
agg = {}
if "income_amount" in num_features:
    agg["income_median"] = ("income_amount", "median")
    agg["income_mean"] = ("income_amount", "mean")
    agg["income_zero_pct"] = ("income_amount", lambda s: round((s == 0).mean() * 100, 1))
    agg["income_missing_pct"] = ("income_amount", lambda s: round(s.isna().mean() * 100, 1))
for f in inputs_count:
    agg[f + "_mean"] = (f, "mean")
if "past_due_rent" in num_features:
    agg["past_due_recorded_pct"] = ("past_due_rent", lambda s: round(s.notna().mean() * 100, 1))
    agg["past_due_median_if_recorded"] = ("past_due_rent", "median")

num_profile = analysis.groupby("cluster_name").agg(**agg).round(1)
num_profile.to_csv("outputs/cluster_numeric_profile.csv")
print(num_profile.T.to_string())

In [ ]:
# category proportions (% of each cluster) for every categorical input
for f in cat_features:
    tab = pd.crosstab(analysis["cluster_name"], analysis[f], normalize="index") * 100
    print(f, "(% of cluster)")
    print(tab.round(1).to_string())
    print()

if len(cat_features) > 0:
    fig, axes = plt.subplots(1, len(cat_features), figsize=(5.5 * len(cat_features), 4.5), squeeze=False)
    for i, f in enumerate(cat_features):
        tab = pd.crosstab(analysis["cluster_name"], analysis[f], normalize="index") * 100
        tab.plot(kind="bar", stacked=True, ax=axes[0][i], width=0.8)
        axes[0][i].set_title(f)
        axes[0][i].set_ylabel("% of cluster")
        axes[0][i].set_xlabel("")
        axes[0][i].legend(fontsize=7, loc="upper right")
    plt.tight_layout()
    plt.savefig("figures/fig_cluster_categories.png", dpi=150)
    plt.show()

In [ ]:
# describe-only fields: urgency proxy and sensitive fields (never used to form clusters)
print("urgent_housing_proxy by cluster (counts):")
print(pd.crosstab(analysis["cluster_name"], analysis["urgent_housing_proxy"].fillna("Unknown"), margins=True))
print()
print("urgent_housing_proxy by cluster (% of cluster):")
print((pd.crosstab(analysis["cluster_name"], analysis["urgent_housing_proxy"].fillna("Unknown"), normalize="index") * 100).round(1))
print()

sensitive_cols = [c for c, r in roles.items() if "sensitive" in r and c in analysis.columns]
if len(sensitive_cols) == 0:
    keys = ["age", "gender", "race", "ethnic", "language", "health"]
    sensitive_cols = [c for c in analysis.columns if any(k in c.lower() for k in keys) and c not in num_features + cat_features]
print("sensitive fields (describe only):", sensitive_cols)
print()

for c in sensitive_cols:
    if mostly_numeric(c):
        vals = pd.to_numeric(analysis[c], errors="coerce")
        print(c, "median by cluster:")
        print(vals.groupby(analysis["cluster_name"]).median().round(1).to_string())
    elif analysis[c].nunique() <= 12:
        print(c, "(% of cluster)")
        print((pd.crosstab(analysis["cluster_name"], analysis[c].fillna("Unknown"), normalize="index") * 100).round(1).to_string())
    print()

**Answer:** _One short description per cluster with its neutral name. Then fill cluster_names above and re-run._

## Policy Questions 1–4

**Question 1 — Different patterns of need:** Do the intake records reveal distinct groups based on housing instability, income/employment circumstances, and household composition? What are the defining characteristics of each group, and how stable are they across the two clustering methods?

**Answer:** _Use the Task 2 profiles, the seed stability, and the K-Means vs hierarchical comparison._

**Question 2 — Overlapping pressures:** Which challenges tend to appear together? Show the relevant within-cluster comparisons; high_arrears_3000 can be one of them. Do not infer cause and effect.

In [ ]:
# high arrears ($3,000+) by cluster, only among units with past-due rent recorded
arrears = analysis["high_arrears_3000"]
recorded = arrears.notna() & (arrears.astype(str).str.strip() != "")
arrears_flag = to_bool(arrears)

q10 = pd.DataFrame({
    "units": analysis.groupby("cluster_name").size(),
    "arrears_recorded": recorded.groupby(analysis["cluster_name"]).sum(),
    "high_arrears_3000": (arrears_flag & recorded).groupby(analysis["cluster_name"]).sum()
})
q10["high_arrears_pct_of_recorded"] = (q10["high_arrears_3000"] / q10["arrears_recorded"] * 100).round(1)
q10.to_csv("outputs/q10_arrears_by_cluster.csv")
print(q10.to_string())

In [ ]:
# employment by housing inside each cluster (counts)
if "employment_status" in cat_features and "housing_status" in cat_features:
    for name, group in analysis.groupby("cluster_name"):
        print(name, "(n = " + str(len(group)) + ")")
        print(pd.crosstab(group["employment_status"], group["housing_status"], margins=True))
        print()

In [ ]:
# employment by $0 income inside each cluster (counts)
if "employment_status" in cat_features and "income_amount" in num_features:
    income_group = np.where(analysis["income_amount"].isna(), "income missing",
                            np.where(analysis["income_amount"] == 0, "$0 income", "income > $0"))
    print(pd.crosstab([analysis["cluster_name"], analysis["employment_status"]], income_group))

**Answer:** _What combinations show up together in each cluster, with counts. Association only._

**Question 3 — Repeat needs and missing information:** Does the distribution of repeat PID contacts (and of linked care-path sub-tickets) differ across the identified patterns? Report counts alongside percentages. PID is name-matched: some sub-tickets carry a different PID from their parent profile. What would you still need to know to distinguish immediate crisis from recurring instability or to evaluate whether assistance improved outcomes?

In [ ]:
# repeat contacts by cluster (counts and %)
q11 = pd.DataFrame({
    "units": analysis.groupby("cluster_name").size(),
    "repeat_units": (analysis["n_profiles"] > 1).groupby(analysis["cluster_name"]).sum()
})
q11["repeat_pct"] = (q11["repeat_units"] / q11["units"] * 100).round(1)
print(q11.to_string())

In [ ]:
# link care-path sub-tickets to units through their parent profile's issue key
parent_cols = [c for c in subs.columns if "parent" in c.lower()]
print("sub-ticket parent column:", parent_cols)

if len(parent_cols) > 0:
    parent_col = parent_cols[0]
    key_to_unit = profiles[["issue_key", "pid"]].copy()
    key_to_unit["unit_id"] = key_to_unit["pid"].astype(str)
    missing = key_to_unit["pid"].isna()
    key_to_unit.loc[missing, "unit_id"] = "ticket_" + key_to_unit.loc[missing, "issue_key"].astype(str)
    key_to_unit = dict(zip(key_to_unit["issue_key"], key_to_unit["unit_id"]))

    subs = subs.copy()
    subs["unit_id"] = subs[parent_col].map(key_to_unit)
    print("sub-tickets:", len(subs))
    print("linked to a profile:", subs["unit_id"].notna().sum())
    print("linked to a unit in the analysis table:", subs["unit_id"].isin(analysis["unit_id"]).sum())

    # name-matched pid on the sub-ticket vs its parent's pid
    if "pid" in subs.columns:
        parent_pid = subs[parent_col].map(dict(zip(profiles["issue_key"], profiles["pid"])))
        both = subs["pid"].notna() & parent_pid.notna()
        print("sub-tickets whose pid differs from parent profile pid:", (subs.loc[both, "pid"].astype(str) != parent_pid[both].astype(str)).sum())

    sub_counts = subs.groupby("unit_id").size().rename("n_subtickets")
    analysis["n_subtickets"] = analysis["unit_id"].map(sub_counts).fillna(0).astype(int)

    q11["units_with_subticket"] = (analysis["n_subtickets"] > 0).groupby(analysis["cluster_name"]).sum()
    q11["subticket_pct"] = (q11["units_with_subticket"] / q11["units"] * 100).round(1)
    q11["total_subtickets"] = analysis.groupby("cluster_name")["n_subtickets"].sum()
    q11.to_csv("outputs/q11_repeat_and_subtickets_by_cluster.csv")
    print()
    print(q11.to_string())

    type_cols = [c for c in subs.columns if c == "issue_type"] + [c for c in subs.columns if ("type" in c.lower() or "path" in c.lower()) and c != "issue_type" and "parent" not in c.lower()]
    if len(type_cols) > 0:
        type_col = type_cols[0]
        linked = subs[subs["unit_id"].isin(analysis["unit_id"])].copy()
        linked["cluster_name"] = linked["unit_id"].map(dict(zip(analysis["unit_id"], analysis["cluster_name"])))
        print()
        print("sub-ticket type (" + type_col + ") by cluster, counts:")
        print(pd.crosstab(linked["cluster_name"], linked[type_col].fillna("Unknown"), margins=True))

**Answer:** _Repeat and sub-ticket differences with counts; what extra data would be needed._

**Question 4 — SLCM program and policy implications:** Propose TWO realistic ways SLCM could investigate or tailor its services. For each, link the recommendation to a plot or statistic and state a limitation. Explain why your clusters should not be used by themselves to determine who receives assistance.

In [ ]:
# one-screen recap of the key numbers
print("unit:", unit)
print("rows: start", flow[0][1], "-> final", len(analysis))
print("clustering inputs:", inputs_money + inputs_count + inputs_flag + inputs_cat)
print("chosen k:", chosen_k)
print("k-means silhouette:", round(silhouette_score(x, km_labels), 3))
print("hierarchical silhouette:", round(silhouette_score(x, hc_labels), 3))
print("agreement (adjusted rand index):", round(adjusted_rand_score(km_labels, hc_labels), 3))
print()
print(size_table.to_string())
print()
print("figures saved in figures/, summary tables saved in outputs/ (no case-level records)")

**Answer:** _Recommendation 1 + evidence + limitation; Recommendation 2 + evidence + limitation; why clusters should not decide assistance._

## What Did You Learn?

**Answer:** _About half a page: (a) strongest discovery, (b) an unexpected result or disagreement between methods, (c) what the data cannot answer, (d) one suggestion for future SLCM data collection._